# Notebook 07 — VodChat (LoRA Fine-Tuning de LLM)

**Projeto:** VOD-IA — PUC-Campinas

**Objetivo:** fazer **fine-tuning** de um LLM pequeno open-source para:
1. Gerar **explicações em linguagem natural** sobre recomendações.
2. **Conversar** com o usuário sobre o catálogo (busca semântica, recomendações guiadas).
3. Aceitar consultas livres tipo *"me sugere algo parecido com X mas mais leve"*.

**Modelo base escolhido:** `TinyLlama-1.1B-Chat-v1.0` (cabe em GPU T4 do Colab gratuito).

**Técnica:** LoRA (Low-Rank Adaptation) + quantização 4-bit (QLoRA) → só ~1% dos parâmetros são treinados.

**Requisitos:** Colab com GPU T4 (Runtime → Change runtime type → GPU).

## 1. Setup

In [ ]:
!pip install -q transformers==4.44.2 peft==0.12.0 trl==0.10.1 \
    accelerate==0.33.0 bitsandbytes==0.43.3 \
    datasets pandas numpy pyarrow

In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

random.seed(42); np.random.seed(42); torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('AVISO: Sem GPU, treino sera lento. Habilite GPU em Runtime > Change runtime type')

## 2. Carrega o catálogo (para gerar o dataset)

In [ ]:
DATA_DIR = Path('data')
VODCHAT_DIR = Path('data/vodchat')
MODELS_DIR = Path('models/vodchat')
VODCHAT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

CONTENTS_PATH = DATA_DIR / 'contents.parquet'
INTERACTIONS_PATH = DATA_DIR / 'interactions.parquet'

if not CONTENTS_PATH.exists():
    raise RuntimeError('Rode primeiro o notebook 06 para gerar contents.parquet')

contents_df = pd.read_parquet(CONTENTS_PATH)
print(f'Catalogo: {len(contents_df)} conteudos')
contents_df.head()

## 3. Geração do dataset de instrução-resposta

Construímos exemplos diversos cobrindo as 3 capacidades-alvo do VodChat. Tudo é gerado **programaticamente** a partir dos metadados reais do catálogo — não dependemos de nenhum LLM externo.

In [ ]:
# Helpers
def list_titles_with_genres(content_ids):
    out = []
    for cid in content_ids:
        row = contents_df[contents_df.content_id == cid]
        if row.empty:
            continue
        out.append((row.iloc[0]['title'], row.iloc[0]['genres']))
    return out

def random_user_history(min_n=3, max_n=8):
    """Simula um historico tematico (com vies para 1-2 generos)."""
    main_genre = random.choice(contents_df['genres'].explode().unique().tolist())
    pool = contents_df[contents_df['genres'].apply(lambda g: main_genre in g)]
    if len(pool) < min_n:
        pool = contents_df
    n = random.randint(min_n, max_n)
    sample = pool.sample(min(n, len(pool)))
    return sample['content_id'].tolist(), main_genre

def random_recommendation(history, main_genre):
    """Simula uma recomendacao plausivel — mesmo genero, conteudo nao visto."""
    pool = contents_df[contents_df['genres'].apply(lambda g: main_genre in g)]
    pool = pool[~pool['content_id'].isin(history)]
    if pool.empty:
        pool = contents_df[~contents_df['content_id'].isin(history)]
    return pool.sample(1).iloc[0]

In [ ]:
# Template 1: EXPLICACAO DE RECOMENDACAO
def make_explanation_example():
    history, main_genre = random_user_history(4, 8)
    titles = list_titles_with_genres(history)
    rec = random_recommendation(history, main_genre)

    hist_txt = ', '.join([f'"{t}"' for t, _ in titles])
    rec_title = rec['title']
    rec_genres = rec['genres']

    instruction = (
        f"O usuario assistiu recentemente: {hist_txt}. "
        f"Explique de forma natural por que recomendariamos \"{rec_title}\" para ele."
    )
    if main_genre in rec_genres:
        response = (
            f"Recomendaria *{rec_title}* porque o usuario tem mostrado forte interesse "
            f"em {main_genre}, e *{rec_title}* combina exatamente esse genero "
            f"({' e '.join(rec_genres)}). Considerando o padrao de consumo recente, "
            f"e provavel que ele tambem gostaria deste titulo."
        )
    else:
        response = (
            f"*{rec_title}* foi recomendado como uma sugestao para diversificar um pouco "
            f"o catalogo do usuario, que tem preferencia por {main_genre}. "
            f"Este titulo traz elementos de {', '.join(rec_genres)}, oferecendo uma experiencia diferente, "
            f"mas ainda assim alinhada com o gosto por narrativas envolventes."
        )
    return {'instruction': instruction, 'response': response}

# Template 2: RECOMENDACAO LIVRE
def make_recommendation_example():
    history, main_genre = random_user_history(3, 6)
    titles = list_titles_with_genres(history)
    hist_txt = ', '.join([f'"{t}"' for t, _ in titles])

    candidates = (contents_df[contents_df['genres'].apply(lambda g: main_genre in g)]
                   .sample(min(3, len(contents_df))))
    rec_lines = []
    for _, r in candidates.iterrows():
        if r['content_id'] in history:
            continue
        rec_lines.append(f'- *{r["title"]}* ({" / ".join(r["genres"])})')

    instruction = (
        f"O usuario assistiu: {hist_txt}. "
        f"Sugira 3 novos titulos do catalogo que ele provavelmente gostaria."
    )
    response = (
        f"Com base no historico (forte presenca de {main_genre}), sugiro estes 3 titulos:\n\n"
        + '\n'.join(rec_lines[:3]) +
        f"\n\nTodos compartilham o tema {main_genre}, mas trazem variacoes que devem manter o interesse."
    )
    return {'instruction': instruction, 'response': response}

# Template 3: PEDIDO DE MUDANCA DE TOM
def make_mood_change_example():
    history, main_genre = random_user_history(3, 6)
    titles = list_titles_with_genres(history)
    hist_txt = ', '.join([f'"{t}"' for t, _ in titles])

    lighter = ['Comedia', 'Romance', 'Animacao', 'Musical']
    target = random.choice(lighter)
    pool = contents_df[contents_df['genres'].apply(lambda g: target in g)]
    if pool.empty:
        pool = contents_df
    pick = pool.sample(1).iloc[0]

    instruction = (
        f"Estou cansado de assistir {main_genre.lower()} (vi: {hist_txt}). "
        f"Quero algo mais leve hoje. O que voce sugere?"
    )
    response = (
        f"Para uma noite mais leve, recomendo *{pick['title']}* — e do genero {target}, "
        f"bem diferente do tom mais intenso de {main_genre}. "
        f"Vai funcionar bem como pausa do que voce tem visto."
    )
    return {'instruction': instruction, 'response': response}

# Template 4: BUSCA SEMANTICA
def make_semantic_search_example():
    target_genre = random.choice(contents_df['genres'].explode().unique().tolist())
    pool = contents_df[contents_df['genres'].apply(lambda g: target_genre in g)]
    if len(pool) < 2:
        return None
    picks = pool.sample(min(3, len(pool)))
    titles = ', '.join([f'*{r["title"]}*' for _, r in picks.iterrows()])

    instruction = f"Quero ver algo de {target_genre} hoje. O que voce tem?"
    response = (
        f"Temos varios titulos de {target_genre} no catalogo. Tres opcoes que destaco: {titles}. "
        f"Se quiser, posso restringir mais por subgenero (por exemplo, {target_genre} mais classico, "
        f"ou {target_genre} mais moderno)."
    )
    return {'instruction': instruction, 'response': response}

# Template 5: PERGUNTA SOBRE O PROPRIO PERFIL
def make_profile_question_example():
    history, main_genre = random_user_history(5, 10)
    titles = list_titles_with_genres(history)
    hist_txt = ', '.join([f'"{t}"' for t, _ in titles])

    instruction = f"Olhando o que eu assisti recentemente ({hist_txt}), quais sao meus generos favoritos?"
    response = (
        f"Pelo seu historico, voce tem forte preferencia por **{main_genre}** — "
        f"aparece em quase todos os titulos recentes. Tambem aparecem outros generos como suporte, "
        f"mas {main_genre} e claramente seu principal interesse no momento."
    )
    return {'instruction': instruction, 'response': response}

In [ ]:
N_EXAMPLES = 3000  # ajuste conforme tempo de treino disponivel

GENERATORS = [
    (make_explanation_example, 0.30),
    (make_recommendation_example, 0.30),
    (make_mood_change_example, 0.15),
    (make_semantic_search_example, 0.15),
    (make_profile_question_example, 0.10),
]

examples = []
while len(examples) < N_EXAMPLES:
    gen = random.choices([g for g,_ in GENERATORS], weights=[w for _,w in GENERATORS])[0]
    ex = gen()
    if ex is not None:
        examples.append(ex)

print(f'Exemplos gerados: {len(examples)}')
print('\nAmostra:')
for ex in examples[:3]:
    print(f'\nInstruction: {ex["instruction"][:200]}')
    print(f'Response:    {ex["response"][:200]}')

# Salva o dataset
ds_path = VODCHAT_DIR / 'sft_dataset.jsonl'
with open(ds_path, 'w', encoding='utf-8') as f:
    for ex in examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')
print(f'\nDataset salvo em {ds_path}')

## 4. Carrega o modelo base com quantização 4-bit (QLoRA)

In [ ]:
MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model = prepare_model_for_kbit_training(model)
print(f'Modelo base carregado: {MODEL_ID}')
print(f'Params totais: {sum(p.numel() for p in model.parameters()):,}')

## 5. Configura o LoRA adapter

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Prepara o dataset no formato chat

In [ ]:
SYSTEM_PROMPT = (
    'Voce e o VodChat, um assistente especializado na plataforma de streaming VOD. '
    'Ajuda usuarios a descobrir conteudo, explica recomendacoes e responde duvidas sobre o catalogo. '
    'Seja conciso, claro e amigavel.'
)

def format_chat(ex):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': ex['instruction']},
        {'role': 'assistant', 'content': ex['response']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

raw_ds = Dataset.from_list(examples)
ds = raw_ds.map(format_chat, remove_columns=['instruction', 'response'])
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split['train'], split['test']

print(f'Treino: {len(train_ds)} | Eval: {len(eval_ds)}')
print('\nExemplo de prompt formatado:')
print(train_ds[0]['text'][:500])
print('...')

## 7. Treino com SFTTrainer

In [ ]:
training_args = SFTConfig(
    output_dir=str(MODELS_DIR / 'checkpoints'),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='epoch',
    save_total_limit=2,
    bf16=True,
    optim='paged_adamw_8bit',
    max_seq_length=1024,
    dataset_text_field='text',
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
)

trainer.train()

## 8. Salva o adapter LoRA

In [ ]:
FINAL_DIR = MODELS_DIR / 'vodchat-lora-final'
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

with open(MODELS_DIR / 'VERSION.txt', 'w') as f:
    f.write('vodchat-v1.0.0')

print('Adapter salvo em:', FINAL_DIR)
for p in sorted(FINAL_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size/1024:.1f} KB)')

## 9. Inferência: carrega o adapter sobre o modelo base

In [ ]:
# Libera memoria do trainer
del trainer, model
torch.cuda.empty_cache()

# Recarrega o modelo base e aplica o adapter
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
vodchat = PeftModel.from_pretrained(base_model, str(FINAL_DIR))
vodchat.eval()
print('VodChat carregado para inferencia.')

In [ ]:
def vodchat_generate(user_message: str, system_prompt: str = SYSTEM_PROMPT,
                     max_new_tokens: int = 200, temperature: float = 0.7):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_message},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(vodchat.device)
    with torch.no_grad():
        out = vodchat.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return completion.strip()

## 10. Testes qualitativos

In [ ]:
TEST_PROMPTS = [
    'O usuario assistiu "Vingança Sombra", "Eclipse Tormenta" e "Caçador Lua" (Ação/Aventura). Por que recomendaria "Reino Destino"?',
    'Quero algo mais leve hoje, cansei de drama. O que sugere?',
    'Olhando o que assisti recentemente (Tormenta Estrela, Caçador Crônica, Reino Sombra), quais sao meus generos favoritos?',
    'Tem documentario historico no catalogo?',
    'Sugere 3 filmes parecidos com "Cidade Eclipse" mas com mais comedia.',
]

for i, prompt in enumerate(TEST_PROMPTS, 1):
    print(f'\n[{i}] USER: {prompt}')
    print(f'    VOD: {vodchat_generate(prompt, max_new_tokens=180)}')

## 11. Benchmark de latência

Em GPU T4: ~300-800ms para 100 tokens. Em CPU sem quantização especial: 2-5s. Por isso, em produção marcamos `with_explanation=true` como opcional/assíncrono.

In [ ]:
import time
lats = []
warmup = vodchat_generate('teste', max_new_tokens=20)
for _ in range(5):
    t0 = time.perf_counter()
    _ = vodchat_generate('Sugira 3 filmes de acao para mim', max_new_tokens=100)
    torch.cuda.synchronize() if device.type == 'cuda' else None
    lats.append((time.perf_counter() - t0) * 1000)
lats = np.array(lats)
print(f'Latencia VodChat (100 novos tokens):')
print(f'  mean = {lats.mean():.0f} ms')
print(f'  P95  = {np.percentile(lats, 95):.0f} ms')

## 12. Empacota tudo para deploy

O adapter LoRA é pequeno (~30MB). O modelo base é baixado pelo serviço uma vez. Em produção:
1. Sobe `models/vodchat/vodchat-lora-final/` para o servidor.
2. O AI Service carrega no startup: `PeftModel.from_pretrained(base, adapter_path)`.
3. (Opcional) merge + quantização GGUF para inferência rápida em CPU com llama.cpp.

In [ ]:
import shutil, tarfile
ARCHIVE = MODELS_DIR / 'vodchat-v1.0.0.tar.gz'
with tarfile.open(ARCHIVE, 'w:gz') as tar:
    tar.add(FINAL_DIR, arcname='vodchat-lora-final')
    tar.add(MODELS_DIR / 'VERSION.txt', arcname='VERSION.txt')
print(f'Empacotado: {ARCHIVE}  ({ARCHIVE.stat().st_size/1e6:.1f} MB)')

## 13. (Opcional) Merge do LoRA no modelo base + export GGUF

Para servir em CPU com baixa latência, mergeia o LoRA no modelo base e converte para GGUF (formato de llama.cpp).

```bash
# 1. Merge do LoRA no base (carregar SEM quantizacao)
python - <<EOF
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained('TinyLlama/TinyLlama-1.1B-Chat-v1.0', torch_dtype='auto')
model = PeftModel.from_pretrained(base, 'models/vodchat/vodchat-lora-final')
merged = model.merge_and_unload()
merged.save_pretrained('models/vodchat/vodchat-merged')
AutoTokenizer.from_pretrained('TinyLlama/TinyLlama-1.1B-Chat-v1.0').save_pretrained('models/vodchat/vodchat-merged')
EOF

# 2. Converter para GGUF (precisa do llama.cpp clonado)
git clone https://github.com/ggerganov/llama.cpp && cd llama.cpp
pip install -r requirements.txt
python convert_hf_to_gguf.py ../models/vodchat/vodchat-merged \
    --outfile ../models/vodchat/vodchat-v1.gguf --outtype f16

# 3. Quantizar para Q4_K_M (recomendado: melhor relacao qualidade/tamanho)
./llama-quantize ../models/vodchat/vodchat-v1.gguf \
    ../models/vodchat/vodchat-v1-Q4_K_M.gguf Q4_K_M
```

Resultado: arquivo `.gguf` de ~700MB que roda em CPU com latência <200ms/100 tokens.

## 14. Próximos passos

- Implementar `app/models/vodchat.py` que carrega o adapter e expõe os métodos `explain()` e `chat()`.
- Integrar no `RecommendationOrchestrator` (chamando VodChat só quando `with_explanation=true`).
- Para produção: usar o `.gguf` quantizado + `llama-cpp-python` para servir o VodChat com baixíssima latência em CPU.
- Coletar feedback dos usuários (botão 👍/👎 nas explicações) → dataset para DPO/RLHF futuro.